In [ ]:
"""
  res 11 -> edge ~25 m,   area ~0.0021 km^2  (well under a block, good default)
"""
NYC_BBOX_POLYGON = {
    "type": "Polygon",
    "coordinates": [[
        [40.685287, -74.032502],
        [40.729917, -74.017194],
        [40.757015, -74.013073],
        [40.854736, -73.949949],
        [40.921400, -73.915666],
        [40.901501, -73.852687],
        [40.850075, -73.863868],
        [40.807420, -73.875048],
        [40.774518, -73.869846],
        [40.760613, -73.841737],
        [40.714454, -73.826190],
        [40.705066, -73.834929],
        [40.732332, -73.884179],
        [40.701005, -73.882600],
        [40.653419, -73.899889],
        [40.601056, -74.040702],
        [40.610155, -74.041878],
        [40.637981, -74.046768],
        [40.685287, -74.032502]
    ]]
}

RESOLUTION = 9  # see guide above -- bumped down for this demo run, see note below

# NOTE: the bounding box above includes a lot of water and NJ/Long Island
# land outside the 5 boroughs. At fine resolutions (10+) that produces
# 500k-1M+ cells and multi-hundred-MB files. For finer parcels, either:
#   (a) swap NYC_BBOX_POLYGON for a precise multi-polygon of the 5 boroughs
#       (e.g. from NYC Open Data's borough boundaries GeoJSON), or
#   (b) filter out water cells afterward (H3 has no built-in land/water
#       mask, so this needs an external land polygon to test cells against).
# Resolution 9 below is used just to keep this demo run/output small.


def polyfill(geojson_polygon, res):
    """Return the set of H3 cell indices covering a GeoJSON polygon."""
    # h3-py v4 API
    h3shape = h3.geo_to_h3shape(geojson_polygon)
    return h3.polygon_to_cells(h3shape, res)


def cells_to_geojson(cells):
    """Convert H3 cells into a GeoJSON FeatureCollection of hexagon parcels."""
    features = []
    for cell in cells:
        boundary = h3.cell_to_boundary(cell)  # list of (lat, lng)
        # GeoJSON wants [lng, lat], and the ring must be closed
        ring = [[lng, lat] for lat, lng in boundary]
        ring.append(ring[0])
        lat, lng = h3.cell_to_latlng(cell)
        features.append({
            "type": "Feature",
            "properties": {
                "h3_index": cell,
                "resolution": RESOLUTION,
                "center_lat": lat,
                "center_lng": lng,
                "area_km2": h3.cell_area(cell, unit="km^2"),
            },
            "geometry": {
                "type": "Polygon",
                "coordinates": [ring],
            },
        })
    return {"type": "FeatureCollection", "features": features}


if __name__ == "__main__":
    cells = polyfill(NYC_BBOX_POLYGON, RESOLUTION)
    print(f"Resolution {RESOLUTION}: {len(cells):,} parcels")
    print(f"Each parcel area: ~{h3.average_hexagon_area(RESOLUTION, unit='km^2'):.6f} km^2")

    geojson = cells_to_geojson(cells)
    out_path = "/mnt/user-data/outputs/nyc_h3_parcels.geojson"
    with open(out_path, "w") as f:
        json.dump(geojson, f)
    print(f"Wrote {out_path}")

    # Also dump a lightweight CSV (index + center point + area) which is
    # often more convenient than full GeoJSON for joining to other data.
    csv_path = "/mnt/user-data/outputs/nyc_h3_parcels.csv"
    with open(csv_path, "w") as f:
        f.write("h3_index,center_lat,center_lng,area_km2\n")
        for cell in cells:
            lat, lng = h3.cell_to_latlng(cell)
            f.write(f"{cell},{lat},{lng},{h3.cell_area(cell, unit='km^2')}\n")
    print(f"Wrote {csv_path}")

# ---------------------------------------------------------------------------
# Note on true squares: if hexagons don't work for your downstream use case
# (e.g. you need axis-aligned square blocks), you'd instead build a regular
# grid in a projected CRS (e.g. UTM 18N / EPSG:32618) with a fixed cell size
# in meters, then reproject back to lat/lng. That's a different tool (not
# H3) -- happy to write that version too if squares are a hard requirement.
# ---------------------------------------------------------------------------

In [1]:
import json
import h3

ModuleNotFoundError: No module named 'h3'

In [6]:
"""
Plot a boundary polygon on an interactive map to visually sanity-check it,
and convert it into correctly-ordered GeoJSON ([lng, lat]) along the way.

Just paste your points into RAW_POINTS_LAT_LNG below in (lat, lng) order --
the natural way people usually write coordinates -- and this script will:
  1. Convert them to GeoJSON's required (lng, lat) order.
  2. Draw the polygon + numbered markers on an OpenStreetMap-based map
     (numbers show point order, so you can confirm the outline traces
     correctly and doesn't cross itself).
  3. Save an .html file you can open directly in a browser.
"""

import folium
import json

# Paste points here in (lat, lng) order -- same order/orientation as given.
RAW_POINTS_LAT_LNG = [
    (40.685287, -74.032502),
    (40.729917, -74.017194),
    (40.757015, -74.013073),
    (40.854736, -73.949949),
    (40.921400, -73.915666),
    (40.901501, -73.852687),
    (40.850075, -73.863868),
    (40.807420, -73.875048),
    (40.774518, -73.869846),
    (40.760613, -73.841737),
    (40.714454, -73.826190),
    (40.705066, -73.834929),
    (40.732332, -73.884179),
    (40.701005, -73.882600),
    (40.653419, -73.899889),
    (40.601056, -74.040702),
    (40.610155, -74.041878),
    (40.637981, -74.046768),
    (40.685287, -74.032502),  # closing point, same as first
]

# --- 1. Convert to GeoJSON's (lng, lat) order -------------------------------
geojson_ring = [[lng, lat] for lat, lng in RAW_POINTS_LAT_LNG]

boundary_polygon = {
    "type": "Polygon",
    "coordinates": [geojson_ring],
}


# --- 2. Plot on an interactive map ------------------------------------------
center_lat = sum(p[0] for p in RAW_POINTS_LAT_LNG) / len(RAW_POINTS_LAT_LNG)
center_lng = sum(p[1] for p in RAW_POINTS_LAT_LNG) / len(RAW_POINTS_LAT_LNG)

m = folium.Map(location=[center_lat, center_lng], zoom_start=11, tiles="OpenStreetMap")

# Draw the polygon outline + fill so you can see the overall shape
folium.Polygon(
    locations=RAW_POINTS_LAT_LNG,  # folium wants (lat, lng), so no swap needed here
    color="crimson",
    weight=3,
    fill=True,
    fill_opacity=0.15,
).add_to(m)

# Add numbered markers at each vertex so you can confirm point order
# (helps catch skipped/out-of-order points or accidental self-crossing)
for i, (lat, lng) in enumerate(RAW_POINTS_LAT_LNG):
    folium.CircleMarker(
        location=[lat, lng],
        radius=5,
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.9,
        popup=f"Point {i}: ({lat}, {lng})",
    ).add_to(m)
    folium.map.Marker(
        [lat, lng],
        icon=folium.DivIcon(html=f'<div style="font-size:11px;color:black;font-weight:bold;">{i}</div>')
    ).add_to(m)

out_path = "boundary_check_map.html"
m.save(out_path)

OSError: [Errno 30] Read-only file system: 'boundary_check_map.html'